# Coordinate descent — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Optimize one coordinate at a time while holding the rest fixed
Coordinate descent makes high-dimensional optimization feel like turning one knob at a time. These five examples show an exact coordinate update, cyclic descent on a quadratic, Gauss-Seidel improvement over stale updates, random coordinate progress, and a Lasso soft-threshold update.

In [ ]:
# 1) exact update for one coordinate of a quadratic
A=np.array([[4.,1.],[1.,3.]]); b=np.array([1.,2.]); x=np.array([0.,0.]); j=0; x[j]=(b[j]-A[j,1]*x[1])/A[j,j]
plt.figure(figsize=(5,3)); plt.bar(['x0 after update','x1'],x); plt.title('coordinate 0 minimized exactly')
assert abs(x[0]-0.25)<1e-12

In [ ]:
# 2) cyclic coordinate descent converges
x=np.zeros(2); path=[x.copy()]
for _ in range(12):
    x[0]=(b[0]-A[0,1]*x[1])/A[0,0]; path.append(x.copy())
    x[1]=(b[1]-A[1,0]*x[0])/A[1,1]; path.append(x.copy())
P=np.array(path); sol=np.linalg.solve(A,b); plt.figure(figsize=(5,4)); plt.plot(P[:,0],P[:,1],'-o',ms=2); plt.scatter([sol[0]],[sol[1]],c='r'); plt.title('cyclic path')
assert np.linalg.norm(x-sol)<1e-4

In [ ]:
# 3) fresh Gauss-Seidel-style updates improve quickly
start=np.array([2.,0.]); x=start.copy(); x[0]=(b[0]-A[0,1]*x[1])/A[0,0]; x[1]=(b[1]-A[1,0]*x[0])/A[1,1]
old=start.copy(); old0=(b[0]-A[0,1]*old[1])/A[0,0]; old1=(b[1]-A[1,0]*old[0])/A[1,1]; stale=np.array([old0,old1])
plt.figure(figsize=(5,3)); plt.bar(['fresh error','stale error'],[np.linalg.norm(x-sol),np.linalg.norm(stale-sol)]); plt.title('using fresh coordinates helps')
assert np.linalg.norm(x-sol)<np.linalg.norm(stale-sol)

In [ ]:
# 4) randomized coordinate descent still descends
rng=np.random.default_rng(1); x=np.array([3.,-2.]); vals=[]
for _ in range(30):
    vals.append(0.5*x@A@x-b@x); j=rng.integers(0,2); x[j]=(b[j]-sum(A[j,k]*x[k] for k in range(2) if k!=j))/A[j,j]
plt.figure(figsize=(5,3)); plt.plot(vals); plt.title('random coordinates reduce objective')
assert vals[-1]<vals[0]

In [ ]:
# 5) Lasso coordinate update is soft-thresholding
z=np.linspace(-3,3,200); lam=1.; soft=np.sign(z)*np.maximum(np.abs(z)-lam,0)
plt.figure(figsize=(5,3)); plt.plot(z,soft); plt.axhline(0,c='k',lw=.5); plt.title('soft threshold creates exact zeros')
assert soft[np.argmin(abs(z-0.5))]==0 and abs((np.sign(3)*max(abs(3)-1,0))-2)<1e-12